# Advanced Apex Project 1 – Progress Report Notebook
## N4: Website / App Traffic Forecasting (Web Traffic Time Series Forecasting, Kaggle)

This notebook is the working code behind the Progress Report. It covers:

1. Data loading and profiling
2. Data preparation (missing values, structure checks, cleaning decisions)
3. Exploratory data analysis (trend, seasonality, segments, spikes)
4. Feature engineering
5. Baseline models and a first regression model
6. Summary numbers for the report and next steps

**How to run:** place `train_1.csv` (from the Kaggle competition) in the same folder as this notebook, then use *Kernel → Restart & Run All*. Every figure used in the report is saved to the `figures/` folder, and every number quoted in the report is saved to `results.json` and `metrics.csv`, so the report and notebook always match.

**Libraries:** pandas, numpy, matplotlib, scikit-learn (no other packages needed).

In [ ]:
import os, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

# ---------------- Configuration (change here only) ----------------
DATA_PATH = "train_1.csv"   # Kaggle file: 1 'Page' column + one column per day
N_PAGES = 2000              # pages used for modelling (None = all eligible pages)
MAX_MISSING_SHARE = 0.5     # drop pages with more than 50% of days missing
HORIZON = 7                 # forecast horizon in days (see feature section)
TEST_DAYS = 28              # last 28 days are held out as the test period
SEED = 42
# -------------------------------------------------------------------

os.makedirs("figures", exist_ok=True)
RESULTS = {}   # every number quoted in the report is collected here

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 1. Load and profile the raw data

Each row is one Wikipedia page and each date column holds that page's visits on that day. We first confirm the structure matches the metadata submitted in the Outline.

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"'{DATA_PATH}' not found. Download train_1.csv from the Kaggle competition "
        "'Web Traffic Time Series Forecasting' and place it next to this notebook.")

raw = pd.read_csv(DATA_PATH)
date_cols = [c for c in raw.columns if c != "Page"]
dates = pd.to_datetime(date_cols)
n_pages, n_days = len(raw), len(date_cols)

print(f"Rows (pages): {n_pages:,}")
print(f"Date columns (days): {n_days}  ({dates.min().date()} to {dates.max().date()})")
print(f"Total columns: {raw.shape[1]}")
RESULTS.update(n_pages_full=n_pages, n_days=n_days,
               date_start=str(dates.min().date()), date_end=str(dates.max().date()))
raw.iloc[:5, :6]

### 1.1 Structural checks
Before any analysis we check for duplicated pages, impossible values (negative or fractional visit counts) and how many recorded values are exactly zero.

In [ ]:
vals = raw[date_cols].to_numpy(dtype="float32")
M = np.isnan(vals)                       # True where a value is missing

dup_pages = int(raw["Page"].duplicated().sum())
neg_vals = int(np.nansum(vals < 0))
frac_vals = int(np.nansum(vals % 1 != 0))
zero_share = float(np.nansum(vals == 0) / (~M).sum())

print("Duplicate page labels :", dup_pages)
print("Negative values       :", neg_vals)
print("Non-integer values    :", frac_vals)
print(f"Zeros as share of recorded values: {zero_share:.2%}")
print(f"Overall missing share : {M.mean():.2%}")
RESULTS.update(dup_pages=dup_pages, neg_vals=neg_vals, frac_vals=frac_vals,
               zero_share=zero_share, overall_missing_share=float(M.mean()))

### 1.2 Missing-value profile
In this dataset a blank means "no count recorded". It matters *where* the blanks are:
- **Leading blanks** (before the first value) usually mean the page did not exist yet.
- **Trailing blanks** (after the last value) mean data stops early.
- **Internal gaps** are blanks between recorded values and are the real data-quality issue.

We separate these three because each deserves a different treatment.

In [ ]:
V = ~M
has_any = V.any(axis=1)
first = np.where(has_any, V.argmax(axis=1), n_days)
last = np.where(has_any, n_days - 1 - V[:, ::-1].argmax(axis=1), -1)
leading = first
trailing = np.where(has_any, n_days - 1 - last, 0)
n_missing = M.sum(axis=1)
internal = n_missing - leading - trailing
miss_share = n_missing / n_days

prof = {
    "pages_all_missing": int((~has_any).sum()),
    "pages_any_missing": int((n_missing > 0).sum()),
    "pages_with_leading_gap": int(((leading > 0) & has_any).sum()),
    "pages_with_trailing_gap": int((trailing > 0).sum()),
    "pages_with_internal_gap": int((internal > 0).sum()),
    "pages_over_threshold": int((miss_share > MAX_MISSING_SHARE).sum()),
}
prof["share_pages_any_missing"] = prof["pages_any_missing"] / n_pages
RESULTS.update(prof)
print(pd.Series(prof).to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(miss_share[miss_share > 0], bins=40, color="#2E5597")
ax[0].set_title("Share of days missing (pages with any missing)")
ax[0].set_xlabel("Share of days missing"); ax[0].set_ylabel("Number of pages")
ax[1].plot(dates, M.mean(axis=0) * 100, color="#C0392B")
ax[1].set_title("Share of pages missing, by date"); ax[1].set_ylabel("% of pages")
savefig("fig01_missing_values")

**Observation (write after viewing Figure 1):** *Describe where the missing values are concentrated and what that suggests (e.g. late-created pages vs. random gaps).*

### 1.3 Splitting the `Page` label into usable fields
`Page` packs four pieces of information: `ARTICLE_NAME_domain_access_agent`. Article names can contain underscores, so we split from the **right** (3 splits).

In [ ]:
parts = raw["Page"].str.rsplit("_", n=3, expand=True)
parts.columns = ["article", "domain", "access", "agent"]

def to_lang(d):
    m = re.match(r"^([a-z\-]+)\.wikipedia\.org$", str(d))
    return m.group(1) if m else "media"      # commons.wikimedia.org, www.mediawiki.org

dom_map = {d: to_lang(d) for d in parts["domain"].unique()}
parts["lang"] = parts["domain"].map(dom_map)

print("Languages / sites:", sorted(parts["lang"].unique()))
print("Access types     :", sorted(parts["access"].dropna().unique()))
print("Agents           :", sorted(parts["agent"].dropna().unique()))
print("\nPages per language (full data):")
print(parts["lang"].value_counts().to_string())
RESULTS["pages_per_lang_full"] = parts["lang"].value_counts().to_dict()

## 2. Data preparation

**Decisions and reasoning**

| Issue | Decision | Why |
|---|---|---|
| Pages with no recorded value at all | Drop | Nothing to learn or forecast |
| Pages missing more than 50% of days | Drop | Too little history to build lag features |
| Internal gaps | Fill by linear interpolation | A short gap is best estimated from the days around it |
| Leading / trailing blanks | Fill with 0 | Leading blanks mean the page did not exist (zero traffic); trailing blanks are rare |
| Spikes and very large values | **Keep** | They are real events (news, holidays); removing them would hide what the forecast must handle |
| Very skewed visit counts | Model on `log(1 + visits)` | Pages range from a handful to millions of visits; the log scale makes errors comparable across pages |

Because the full dataset (about 145,000 pages × 550 days) is large, we work with a **random sample of pages** drawn after filtering. The sample keeps all 550 days for each page, so time-series behaviour is not distorted.

In [ ]:
keep = has_any & (miss_share <= MAX_MISSING_SHARE)
eligible = np.flatnonzero(keep)
RESULTS["pages_eligible"] = int(len(eligible))
RESULTS["pages_dropped"] = int(n_pages - len(eligible))
print(f"Eligible pages: {len(eligible):,}   Dropped: {n_pages - len(eligible):,}")

rng = np.random.default_rng(SEED)
if N_PAGES is not None and N_PAGES < len(eligible):
    idx = np.sort(rng.choice(eligible, size=N_PAGES, replace=False))
else:
    idx = eligible
RESULTS["pages_modelled"] = int(len(idx))

meta = parts.iloc[idx].reset_index(drop=True)
meta["page"] = raw["Page"].iloc[idx].values

wide = pd.DataFrame(vals[idx], columns=dates, index=meta["page"].values)
cells_missing_before = int(wide.isna().sum().sum())
wide = wide.interpolate(axis=1, limit_area="inside")     # internal gaps only
cells_after_interp = int(wide.isna().sum().sum())
wide = wide.fillna(0.0)                                  # leading / trailing blanks

RESULTS.update(cells_missing_before=cells_missing_before,
               cells_filled_by_interpolation=cells_missing_before - cells_after_interp,
               cells_filled_with_zero=cells_after_interp)
print(f"Sample size: {wide.shape[0]:,} pages x {wide.shape[1]} days")
print(f"Missing cells before: {cells_missing_before:,}")
print(f"  filled by interpolation: {cells_missing_before - cells_after_interp:,}")
print(f"  filled with zero       : {cells_after_interp:,}")
assert wide.isna().sum().sum() == 0

del raw, vals, M, V   # free memory

## 3. Exploratory data analysis

### 3.1 Overall traffic over time
We add up visits across all sampled pages to get one "total traffic" series. The 7-day rolling mean smooths out the weekly cycle so the longer trend is visible.

In [ ]:
total = wide.sum(axis=0)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(total.index, total.values / 1e6, lw=0.8, alpha=0.6, label="Daily total")
ax.plot(total.index, total.rolling(7).mean() / 1e6, lw=2, color="#C0392B", label="7-day rolling mean")
ax.set_ylabel("Visits (millions, sampled pages)"); ax.set_title("Total daily traffic"); ax.legend()
savefig("fig02_total_traffic")

first_month = total.iloc[:30].mean(); last_month = total.iloc[-30:].mean()
RESULTS.update(total_mean_daily=float(total.mean()), total_max_daily=float(total.max()),
               total_max_date=str(total.idxmax().date()),
               growth_first_to_last_30d_pct=float((last_month / first_month - 1) * 100))
print(f"Average daily total: {total.mean():,.0f}  | peak {total.max():,.0f} on {total.idxmax().date()}")
print(f"Change, first 30 days -> last 30 days: {RESULTS['growth_first_to_last_30d_pct']:+.1f}%")

**Observation (write after viewing Figure 2):** *Is there a long-term trend? Are there visible jumps? Does the level change over the 18 months?*

### 3.2 Weekly pattern
To measure the day-of-week effect without the long-term trend getting in the way, we divide each day's total by the centred 7-day average around it. A value of 1.05 means that weekday is 5% above a typical day in that week.

In [ ]:
ratio = total / total.rolling(7, center=True).mean()
dow_ratio = ratio.groupby(total.index.dayofweek).mean()
dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.bar(dow_names, dow_ratio.values, color="#2E5597")
ax.axhline(1, color="k", lw=0.8); ax.set_ylim(dow_ratio.min() - 0.03, dow_ratio.max() + 0.03)
ax.set_title("Traffic relative to weekly average, by weekday"); ax.set_ylabel("Ratio")
savefig("fig03_weekday_pattern")

RESULTS["dow_ratio"] = {n: float(v) for n, v in zip(dow_names, dow_ratio.values)}
RESULTS["peak_weekday"] = dow_names[int(dow_ratio.values.argmax())]
RESULTS["lowest_weekday"] = dow_names[int(dow_ratio.values.argmin())]
print(dow_ratio.round(3).rename(dict(enumerate(dow_names))).to_string())

**Observation (write after viewing Figure 3):** *Which day is highest and lowest, and how big is the gap? What does this tell you about the features needed (day-of-week, lag of 7 days)?*

### 3.3 Who generates the traffic? (language, device, visitor type)

In [ ]:
page_total = wide.sum(axis=1).values
seg = pd.DataFrame({"lang": meta["lang"], "access": meta["access"],
                    "agent": meta["agent"], "views": page_total})

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, col, title in zip(ax, ["lang", "access", "agent"],
                         ["Share of views by language/site", "Share of views by access type", "Share of views by agent"]):
    s = seg.groupby(col)["views"].sum().sort_values(ascending=False)
    (s / s.sum() * 100).plot.bar(ax=a, color="#2E5597")
    a.set_title(title); a.set_ylabel("% of total views"); a.set_xlabel("")
savefig("fig04_traffic_segments")

for col in ["lang", "access", "agent"]:
    s = seg.groupby(col)["views"].sum()
    RESULTS[f"view_share_{col}"] = (s / s.sum()).round(4).to_dict()
    print(f"\n{col}:\n{(s / s.sum()).sort_values(ascending=False).round(3).to_string()}")

**Observation (write after viewing Figure 4):** *Which language/site dominates? How much traffic is automated (spider)? Does this suggest forecasting segments separately or adding them as features?*

### 3.4 How different are pages from each other?
Page-level averages are extremely uneven. The coefficient of variation (standard deviation ÷ mean) shows how "steady" or "spiky" a page's own traffic is.

In [ ]:
page_mean = wide.mean(axis=1)
cv = (wide.std(axis=1) / page_mean.replace(0, np.nan)).dropna()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(np.log10(page_mean + 1), bins=40, color="#2E5597")
ax[0].set_title("Average daily visits per page (log10 scale)"); ax[0].set_xlabel("log10(1 + mean visits)")
ax[0].set_ylabel("Number of pages")
ax[1].hist(cv.clip(upper=cv.quantile(0.99)), bins=40, color="#C0392B")
ax[1].set_title("Variability of each page (std / mean)"); ax[1].set_xlabel("Coefficient of variation")
savefig("fig05_page_distribution")

RESULTS.update(page_mean_median=float(page_mean.median()), page_mean_mean=float(page_mean.mean()),
               page_mean_max=float(page_mean.max()), page_cv_median=float(cv.median()))
print(f"Median page: {page_mean.median():,.0f} visits/day | mean page: {page_mean.mean():,.0f} | max page: {page_mean.max():,.0f}")
print(f"Median coefficient of variation: {cv.median():.2f}")

top10 = page_mean.sort_values(ascending=False).head(10).round(0)
top10.to_csv("top10_pages.csv")
print("\nTop 10 pages by average daily visits:")
print(top10.to_string())

**Observation (write after viewing Figure 5):** *The mean is far above the median – what does this say about how a few pages dominate traffic, and why does it justify modelling on a log scale?*

### 3.5 Language trends

In [ ]:
by_lang = wide.groupby(meta["lang"].values).sum().T
top_langs = by_lang.sum().sort_values(ascending=False).head(5).index
fig, ax = plt.subplots(figsize=(11, 3.8))
for l in top_langs:
    ax.plot(by_lang.index, by_lang[l].rolling(7).mean(), label=l, lw=1.5)
ax.set_yscale("log"); ax.legend(ncol=5); ax.set_title("7-day average traffic by language (log scale)")
ax.set_ylabel("Visits")
savefig("fig06_language_trends")

**Observation (write after viewing Figure 6):** *Do languages move together or differently? Are there language-specific peaks?*

### 3.6 Seasonality check with autocorrelation
Autocorrelation measures how strongly today's change resembles the change *k* days earlier. We use day-to-day **changes** of the log total so the slow trend does not hide the pattern. Peaks at lags 7, 14, 21... indicate a weekly cycle.

In [ ]:
s_diff = np.log(total.replace(0, np.nan)).diff().dropna()
acf = np.array([s_diff.autocorr(l) for l in range(1, 43)])
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(range(1, 43), acf, color=["#C0392B" if l % 7 == 0 else "#2E5597" for l in range(1, 43)])
ax.set_xlabel("Lag (days)"); ax.set_ylabel("Autocorrelation")
ax.set_title("Autocorrelation of daily change in log total traffic (red = multiples of 7)")
savefig("fig07_autocorrelation")
RESULTS.update(acf_lag1=float(acf[0]), acf_lag7=float(acf[6]), acf_lag14=float(acf[13]))
print(f"ACF at lag 1: {acf[0]:.2f} | lag 7: {acf[6]:.2f} | lag 14: {acf[13]:.2f}")

**Observation (write after viewing Figure 7):** *Which lags stand out and what does that imply for the lag features we include?*

### 3.7 Spikes and unusual days
A spike is a day where total traffic is far from its local normal level. We compare each day with the 15-day rolling median around it.

In [ ]:
base = total.rolling(15, center=True, min_periods=8).median()
dev = (total / base - 1)
top_spikes = dev.abs().sort_values(ascending=False).head(5).index
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(dev.index, dev.values * 100, lw=0.9, color="#2E5597")
ax.scatter(top_spikes, dev.loc[top_spikes] * 100, color="#C0392B", zorder=3, label="Top 5 unusual days")
ax.set_ylabel("% above/below local median"); ax.set_title("Unusual days in total traffic"); ax.legend()
savefig("fig08_spikes")
RESULTS["top_spike_dates"] = {str(d.date()): float(dev.loc[d] * 100) for d in top_spikes}
print(pd.Series({str(d.date()): round(dev.loc[d] * 100, 1) for d in top_spikes}).to_string())

**Observation (write after viewing Figure 8):** *Do the unusual days look like real events (holidays, news) or data problems? How might they affect a regression model?*

## 4. Feature engineering

### 4.1 Forecasting set-up and why it matters
We frame the task as **"predict each page's visits `HORIZON` days ahead"** (default 7 days). A feature may only use information that would be known `HORIZON` days before the target day. This is the main guard against **data leakage**: every history-based feature is shifted by at least `HORIZON` days.

| Feature group | Features | Reasoning |
|---|---|---|
| Lags | visits 7, 14, 21, 28 days earlier | Weekly cycle (ACF peaks at multiples of 7) |
| Rolling and smoothed level | 7- and 28-day rolling mean, 7-day rolling std, 7- and 28-day EWMA | Recent level and how noisy the page is; EWMA gives more weight to recent days |
| Trend | 7-day mean minus 28-day mean | Is the page rising or falling lately? |
| Calendar | day of week (Tue–Sun, Monday as reference) | Weekday effect found in EDA |
| Page attributes | access type, agent, language (top 8) | Different segments behave differently |

**Deliberately left out for now:** month/season features (the data covers only 18 months, so each month appears at most twice and cannot be separated from noise) and a plain time counter (a linear model would extrapolate it wrongly). Both are listed under next steps.

All history features are built on the log scale: `log(1 + visits)`.

In [ ]:
L = np.log1p(wide.T)          # rows = dates, columns = pages
T, P = L.shape
lags = [HORIZON + 7 * k for k in range(4)]
hist = L.shift(HORIZON)       # everything below only sees data >= HORIZON days old

F = {f"lag_{l}": L.shift(l) for l in lags}
F["roll_mean_7"] = hist.rolling(7).mean()
F["roll_mean_28"] = hist.rolling(28).mean()
F["roll_std_7"] = hist.rolling(7).std()
F["ewma_7"] = hist.ewm(span=7, adjust=False).mean()
F["ewma_28"] = hist.ewm(span=28, adjust=False).mean()
F["trend_gap"] = F["roll_mean_7"] - F["roll_mean_28"]

# flatten to long format (one row = one page on one date)
X = {k: v.to_numpy(dtype="float32").reshape(-1) for k, v in F.items()}
y = L.to_numpy(dtype="float32").reshape(-1)
date_idx = np.repeat(np.arange(T), P)
page_idx = np.tile(np.arange(P), T)

# calendar features
dow = dates.dayofweek.to_numpy()
dow_cols = []
for d, name in zip(range(1, 7), dow_names[1:]):
    X[f"dow_{name}"] = (dow[date_idx] == d).astype("float32"); dow_cols.append(f"dow_{name}")

# page attribute features
page_attr = {"access": meta["access"].to_numpy(), "agent": meta["agent"].to_numpy(), "lang": meta["lang"].to_numpy()}
attr_cols = []
for val in ["desktop", "mobile-web"]:
    X[f"access_{val}"] = (page_attr["access"][page_idx] == val).astype("float32"); attr_cols.append(f"access_{val}")
X["agent_spider"] = (page_attr["agent"][page_idx] == "spider").astype("float32"); attr_cols.append("agent_spider")
top8 = seg.groupby("lang")["views"].sum().sort_values(ascending=False).head(8).index
for l in top8:
    X[f"lang_{l}"] = (page_attr["lang"][page_idx] == l).astype("float32"); attr_cols.append(f"lang_{l}")

lag_cols = [f"lag_{l}" for l in lags]
roll_cols = ["roll_mean_7", "roll_mean_28", "roll_std_7", "ewma_7", "ewma_28", "trend_gap"]
FEATURE_SETS = {
    "A: lags only": lag_cols,
    "B: + rolling/EWMA/trend": lag_cols + roll_cols,
    "C: + calendar": lag_cols + roll_cols + dow_cols,
    "D: + page attributes (full)": lag_cols + roll_cols + dow_cols + attr_cols,
}
print(f"Total features in full set: {len(FEATURE_SETS['D: + page attributes (full)'])}")
RESULTS["n_features_full"] = len(FEATURE_SETS["D: + page attributes (full)"])
RESULTS["feature_groups"] = {k: len(v) for k, v in FEATURE_SETS.items()}

### 4.2 Train / test split by time
Random splitting would let the model "see the future". Instead the **last 28 days** are held out as the test period and all earlier days are used for training. The first rows of every page are dropped because their lag and rolling features do not have enough history yet.

In [ ]:
START = HORIZON + 27                      # first date with all rolling features available
test_start = T - TEST_DAYS
tr = (date_idx >= START) & (date_idx < test_start)
te = date_idx >= test_start

for c in FEATURE_SETS["D: + page attributes (full)"]:
    assert np.isfinite(X[c][tr | te]).all(), f"NaN in {c}"

print(f"Train period: {dates[START].date()} -> {dates[test_start-1].date()}  ({tr.sum():,} rows)")
print(f"Test period : {dates[test_start].date()} -> {dates[-1].date()}  ({te.sum():,} rows)")
RESULTS.update(train_start=str(dates[START].date()), train_end=str(dates[test_start - 1].date()),
               test_start=str(dates[test_start].date()), test_end=str(dates[-1].date()),
               train_rows=int(tr.sum()), test_rows=int(te.sum()), horizon=HORIZON)

## 5. Baseline models

We compare three levels of sophistication, from "no model" to a first regression:

1. **Seasonal naive** – predict that a page gets the same visits as the same weekday last week (the lag-7 value).
2. **Moving average** – predict the average of the last 7 available days.
3. **Linear regression** – fitted on the engineered features, predicting `log(1 + visits)`.

**Metrics** (computed on the original visit scale after converting back):
- **SMAPE** – the Kaggle competition metric: average percentage error, symmetric and bounded; 0 is perfect.
- **MAE** – average absolute error in visits.
- **RMSE** – like MAE but punishes large misses more, so it is dominated by the biggest pages.
- **MAE (log)** – error on the log scale, roughly a relative error that treats small and large pages fairly.

In [ ]:
def smape(a, p):
    denom = np.abs(a) + np.abs(p)
    out = np.where(denom == 0, 0.0, 2 * np.abs(a - p) / np.where(denom == 0, 1, denom))
    return 100 * out.mean()

def evaluate(y_log, p_log):
    a = np.expm1(y_log); p = np.clip(np.expm1(p_log), 0, None)
    return {"SMAPE_%": smape(a, p), "MAE": np.abs(a - p).mean(),
            "RMSE": float(np.sqrt(((a - p) ** 2).mean())),
            "MAE_log": float(np.abs(y_log - np.clip(p_log, 0, None)).mean())}

y_tr, y_te = y[tr], y[te]
rows = {}
rows["Seasonal naive (same weekday last week)"] = evaluate(y_te, X[lag_cols[0]][te])
rows["Moving average (7 days)"] = evaluate(y_te, X["roll_mean_7"][te])

models, preds = {}, {}
for name, cols in FEATURE_SETS.items():
    Xtr = np.column_stack([X[c][tr] for c in cols]); Xte = np.column_stack([X[c][te] for c in cols])
    m = LinearRegression().fit(Xtr, y_tr)
    models[name] = (m, cols); preds[name] = m.predict(Xte)
    rows[f"Linear regression {name}"] = evaluate(y_te, preds[name])

metrics = pd.DataFrame(rows).T.round(3)
metrics.to_csv("metrics.csv")
RESULTS["metrics"] = {k: {kk: float(vv) for kk, vv in v.items()} for k, v in metrics.iterrows()}
metrics

### 5.1 Model comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
order = metrics["SMAPE_%"]
cols_ = ["#999999", "#999999"] + ["#2E5597"] * (len(order) - 2)
ax.barh(order.index[::-1], order.values[::-1], color=cols_[::-1])
ax.set_xlabel("SMAPE % (lower is better)"); ax.set_title("Test-period error by model")
for i, v in enumerate(order.values[::-1]):
    ax.text(v + 0.2, i, f"{v:.1f}", va="center", fontsize=8)
savefig("fig09_model_comparison")

FULL = "D: + page attributes (full)"
best_name = f"Linear regression {FULL}"
RESULTS["smape_naive"] = float(metrics.loc["Seasonal naive (same weekday last week)", "SMAPE_%"])
RESULTS["smape_best_lr"] = float(metrics.loc[best_name, "SMAPE_%"])
RESULTS["improvement_vs_naive_pct_points"] = RESULTS["smape_naive"] - RESULTS["smape_best_lr"]
print(f"Seasonal naive SMAPE: {RESULTS['smape_naive']:.2f}  | Full linear regression SMAPE: {RESULTS['smape_best_lr']:.2f}")

**Observation (write after viewing Figure 9 and the table):** *Does the regression beat the simple baselines? By how much? Does the ranking by SMAPE agree with the ranking by RMSE and MAE (log)? If not, why might that be?*

### 5.2 Which feature groups helped? (ablation)
Adding feature groups one at a time shows which ones actually improve the forecast.

In [ ]:
abl = metrics.loc[[f"Linear regression {k}" for k in FEATURE_SETS]]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar([k.split(":")[0] for k in FEATURE_SETS], abl["SMAPE_%"].values, color="#2E5597")
ax.set_ylabel("SMAPE %"); ax.set_title("Effect of adding feature groups (A->D)")
ax.set_ylim(abl["SMAPE_%"].min() * 0.97, abl["SMAPE_%"].max() * 1.03)
for i, v in enumerate(abl["SMAPE_%"].values):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
savefig("fig10_feature_ablation")
print(abl[["SMAPE_%", "MAE_log"]].to_string())

**Observation (write after viewing Figure 10):** *Which group produced the largest improvement and which produced little or none? What would you keep, drop or rethink?*

### 5.3 Forecast vs. actual (test period)
We add up predictions across all sampled pages for each test day and compare with the actual total.

In [ ]:
d_te = date_idx[te] - test_start
actual_tot = np.bincount(d_te, weights=np.expm1(y_te), minlength=TEST_DAYS)
naive_tot = np.bincount(d_te, weights=np.clip(np.expm1(X[lag_cols[0]][te]), 0, None), minlength=TEST_DAYS)
lr_tot = np.bincount(d_te, weights=np.clip(np.expm1(preds[FULL]), 0, None), minlength=TEST_DAYS)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(dates[test_start:], actual_tot / 1e6, "k-", lw=2, label="Actual")
ax.plot(dates[test_start:], naive_tot / 1e6, "--", color="#999999", label="Seasonal naive")
ax.plot(dates[test_start:], lr_tot / 1e6, color="#C0392B", lw=1.8, label="Linear regression (full)")
ax.set_ylabel("Visits (millions, sampled pages)"); ax.set_title("Total traffic: forecast vs. actual, test period")
ax.legend()
savefig("fig11_forecast_vs_actual")

agg_smape_lr = smape(actual_tot, lr_tot); agg_smape_naive = smape(actual_tot, naive_tot)
RESULTS.update(aggregate_smape_lr=float(agg_smape_lr), aggregate_smape_naive=float(agg_smape_naive))
print(f"SMAPE on aggregate total series  -> naive: {agg_smape_naive:.2f}  | linear regression: {agg_smape_lr:.2f}")

**Observation (write after viewing Figure 11):** *Does the forecast follow the weekly shape and the level? Are there days where it clearly misses?*

*Note: the aggregate error is usually much lower than the page-level error because over- and under-predictions of individual pages partly cancel when added together. Compare the two numbers above and explain the gap in your report.*

### 5.4 Where does the model struggle?
Error by segment shows whether some kinds of pages are much harder to forecast.

In [ ]:
tp = page_idx[te]
errdf = pd.DataFrame({"lang": page_attr["lang"][tp], "access": page_attr["access"][tp],
                      "agent": page_attr["agent"][tp],
                      "a": np.expm1(y_te), "p": np.clip(np.expm1(preds[FULL]), 0, None)})
seg_err = {c: errdf.groupby(c).apply(lambda g: smape(g["a"].values, g["p"].values)) for c in ["lang", "access", "agent"]}

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6), gridspec_kw={"width_ratios": [3, 1.5, 1]})
for a, c in zip(ax, ["lang", "access", "agent"]):
    seg_err[c].sort_values().plot.bar(ax=a, color="#2E5597"); a.set_title(f"SMAPE by {c}"); a.set_xlabel("")
savefig("fig12_error_by_segment")
RESULTS["smape_by_segment"] = {c: seg_err[c].round(3).to_dict() for c in seg_err}
for c in seg_err:
    print(f"\nSMAPE by {c}:\n{seg_err[c].sort_values().round(2).to_string()}")

**Observation (write after viewing Figure 12):** *Which segments are hardest and why (e.g. low-traffic pages where one visit changes the percentage a lot, spiders, a particular language)?*

### 5.5 What the regression learned
The coefficients show which inputs the model relies on. Several lag features measure similar things, so individual coefficients should be read as a rough guide rather than exact effects.

In [ ]:
m_full, cols_full = models[FULL]
coef = pd.Series(m_full.coef_, index=cols_full)
top_coef = coef.reindex(coef.abs().sort_values(ascending=False).index).head(12)
print(top_coef.round(3).to_string())
RESULTS["top_coefficients"] = top_coef.round(4).to_dict()
RESULTS["intercept"] = float(m_full.intercept_)

resid = y_te - preds[FULL]
RESULTS.update(resid_mean_log=float(resid.mean()), resid_std_log=float(resid.std()))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
top_coef[::-1].plot.barh(ax=ax[0], color="#2E5597"); ax[0].set_title("Largest coefficients (full model)")
ax[1].hist(np.clip(resid, -2, 2), bins=60, color="#C0392B"); ax[1].set_title("Test residuals (log scale, clipped to ±2)")
ax[1].set_xlabel("actual − predicted (log)")
savefig("fig13_coefficients_residuals")
print(f"\nResidual mean: {resid.mean():.3f} | std: {resid.std():.3f}")

**Observation (write after viewing Figure 13):** *Which features carry the most weight? Are residuals centred on zero and roughly symmetric, or is there a bias or heavy tail?*

## 6. Save numbers for the report

In [ ]:
def clean(o):
    if isinstance(o, dict):  return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    return o

with open("results.json", "w") as f:
    json.dump(clean(RESULTS), f, indent=2)

print("Saved: results.json, metrics.csv, top10_pages.csv, and figures/ ->", sorted(os.listdir("figures")))
print("\n=== KEY NUMBERS TO QUOTE IN THE REPORT ===")
for k in ["pages_eligible", "pages_dropped", "pages_modelled", "share_pages_any_missing",
          "pages_with_leading_gap", "pages_with_internal_gap", "train_start", "train_end",
          "test_start", "test_end", "n_features_full", "smape_naive", "smape_best_lr",
          "improvement_vs_naive_pct_points", "aggregate_smape_naive", "aggregate_smape_lr"]:
    print(f"{k:34s}: {RESULTS[k]}")

## 7. Summary, limitations and next steps

**Completed so far**
- Loaded and profiled the full dataset; checked structure and missing values.
- Prepared the data (cleaning rules documented in Section 2) and sampled pages for modelling.
- Explored trend, weekly seasonality, segments, page variability and spikes.
- Built leakage-safe lag, rolling, EWMA, trend, calendar and page-attribute features.
- Fitted three baselines and a linear regression with a feature-group ablation, evaluated on a held-out final 28 days.

**Limitations (be honest in the report)**
- Modelling uses a random sample of pages, not all ~145,000.
- Forecasts are 7-day-ahead using *actual* past data, not a full multi-week recursive forecast.
- Zero-filling leading blanks may blur "page did not exist" with "no visits".
- One train/test split only; no cross-validation yet.
- Linear regression cannot capture non-linear effects or sudden spikes.

**Planned (not yet done)**
- Rolling-origin (time-series) cross-validation instead of a single split.
- Multi-step forecasting up to 60 days ahead.
- Regularised regression (Ridge/Lasso) and tree-based models (e.g. gradient boosting) for comparison.
- Holiday / event features and yearly seasonality, if the data allows.
- Separate treatment for very low-traffic pages and automated (spider) traffic.
- Run on all eligible pages and compare with the sample results.